In [ ]:
%load_ext autoreload
%autoreload 2
%cd ../../..

In [ ]:
import nbformat
from nbconvert import MarkdownExporter
from pathlib import Path
from dataclasses import dataclass

In [ ]:
@dataclass
class NotebookForAgent:
    path: str
    markdown_content: str
    cell_count: int
    code_cells: int
    executed_cells: int
    
    @property
    def execution_rate(self) -> float:
        return (self.executed_cells / self.code_cells * 100) if self.code_cells else 0
    
    @property
    def summary(self) -> str:
        name = Path(self.path).name
        return f"{name}: {self.cell_count} cells ({self.code_cells} code, {self.execution_rate:.0f}% executed)"

def parse_notebook(notebook_path: str) -> NotebookForAgent:
    path = Path(notebook_path).expanduser().resolve()
    with open(path) as f:
        nb = nbformat.read(f, as_version=4)
    
    exporter = MarkdownExporter()
    markdown_content, _ = exporter.from_notebook_node(nb)
    
    cells = nb.cells
    code_cells = [c for c in cells if c.cell_type == 'code']
    executed_cells = [c for c in code_cells if c.execution_count is not None]
    
    return NotebookForAgent(
        path=str(path),
        markdown_content=markdown_content,
        cell_count=len(cells),
        code_cells=len(code_cells),
        executed_cells=len(executed_cells)
    )

notebook_path = 'src/clinical_ai_guardrails/notebooks/1.train.ipynb'
notebook = parse_notebook(notebook_path)

In [ ]:
notebook.summary

In [ ]:
notebook.markdown_content